<a href="https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of analysis: One row represents one content item for one client on one report date.
Time window: I will use March 2026 as the development window, from 2026-03-01 to 2026-03-31.
This grain lets me measure daily search and analytics performance for each content item and client.

In [53]:
# Verify the unit of analysis and time window

march_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id || '|' || client_hash_id || '|' || CAST(report_date AS VARCHAR)) AS unique_content_client_date_rows,
            MIN(report_date) AS first_date,
                MAX(report_date) AS last_date,
                    COUNT(DISTINCT content_hash_id) AS unique_content_items,
                        COUNT(DISTINCT client_hash_id) AS unique_clients
                        FROM fact_content_daily_performance
                        WHERE month = '2026-03'
                        """).df()
display(march_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_content_client_date_rows,first_date,last_date,unique_content_items,unique_clients
0,9841378,9841378,2026-03-01,2026-03-31,331437,55


## 2. Fields: feature / label / context / excluded

### Features

I will use observable signals that are available before the decision point:

- impressions_90d — search visibility/demand
- clicks_90d — search clicks
- sessions_90d — analytics traffic
- ctr — click-through rate
- avg_position — average search position

### Label / proxy

- trend_direction / is_declining_label — used as a proxy for whether the content item is declining.

### Context

- content_age_days — age of the content
- content_type — type of content
- main_intent — search/content intent
- client_hash_id — grouping and validation context
- content_hash_id — content-item identifier

### Excluded

- Product decision scores or flags such as health_score, priority_score, or action_type are excluded because they represent product decisions rather than independent observable signals.
- Raw URLs, client names, raw queries, and other identifying information are excluded because the dataset is designed to remain public-safe.

In [54]:
import duckdb

con = duckdb.connect()

print("DuckDB connection created successfully.")

DuckDB connection created successfully.


In [55]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

I will verify four things from the March 2026 slice:

1. The grain is content item × client × day.
2. The slice contains the expected number of rows.
3. The selected signals have measurable availability.
4. The dates fall inside the intended March 2026 development window.

I will use these checks before building the feature frame so that the data contract is based on observed data rather than assumptions.

In [56]:
march_counts = con.execute("""
SELECT
    COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
            COUNT(DISTINCT client_hash_id) AS unique_clients,
                MIN(report_date) AS first_date,
                    MAX(report_date) AS last_date
                    FROM read_parquet(
                        '/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet'
                        )
                        """).df()
display(march_counts)

,total_rows,unique_content_items,unique_clients,first_date,last_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [57]:
availability_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
            COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
            FROM read_parquet(
                '/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet'
                )
                """).df()
display(availability_check)

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## 4. Data limits

### Data limitations

This dataset supports observational and decision-support analysis, but it cannot prove causality.

The history is unbalanced, so different clients and content items can have different amounts of historical data.

Some early rows may have GSC data but no GA4 data because analytics tracking was not available yet. Therefore, missing GA4 values must not automatically be interpreted as zero traffic.

The March 2026 slice is useful for measured and directional analysis, but it has important limits.

- The history is limited to the available warehouse window, so it cannot represent every possible seasonal or long-term pattern.
- GSC and GA4 availability is uneven, so some rows cannot support metrics from those sources.
- The data is observational, so it can show associations and support decisions, but it cannot prove that one content change caused a performance change.
- The March 2026 window is one month, so it may not capture longer-term content trends.
- The final June 2026 month is excluded from development because it is the natural outcome window for past-to-future labels.

In [58]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.